# Part 1: Environment Setup

### Requirements
- **Python:** 3.9+
- **Hardware:** Works on CPU (Colab/Jupyter standard).
- **API Key:** You will need a Groq API Key (get one at [console.groq.com](https://console.groq.com/)).

We install `groq` for inference, `chromadb` for our vector store, and `sentence-transformers` for local embeddings. `langchain` and `langchain-community` act as the glue for the pipeline.

In [54]:
!pip install -qU groq chromadb sentence-transformers langchain langchain-community langchain-text-splitters pypdf python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.5 MB/s eta 0:00:00


In [55]:
import sys
import groq
import chromadb
import langchain
import sentence_transformers

print(f"--- Validation Success ---")
print(f"Python version: {sys.version.split(' ')[0]}")
print(f"Groq version: {groq.__version__}")
print(f"ChromaDB version: {chromadb.__version__}")
print(f"LangChain version: {langchain.__version__}")
print(f"Sentence-Transformers version: {sentence_transformers.__version__}")

--- Validation Success ---
Python version: 3.12.13
Groq version: 1.1.2
ChromaDB version: 1.5.6
LangChain version: 1.2.15
Sentence-Transformers version: 5.3.0


# Part 2: System Architecture

Our RAG pipeline follows these steps:
1. **Document Loading:** Ingesting PDF or Text FAQ documents.
2. **Chunking:** Splitting text into manageable pieces (500 tokens) with overlap (50 tokens) to maintain context.
3. **Embedding:** Using `all-MiniLM-L6-v2` (a fast, local model) to turn text into vectors.
4. **Indexing:** Storing vectors in **ChromaDB** for semantic search.
5. **Retrieval:** Fetching the Top-K most relevant chunks based on user queries.
6. **Generation:** Passing the chunks + query to **Groq (Llama-3)** for a grounded response.

In [56]:
import os
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from groq import Groq

# Securely fetch Groq API Key
# Ensure you have 'GROQ_API_KEY' set in your Colab Secrets (🔑 icon)
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    client = Groq(api_key=GROQ_API_KEY)
    print("Groq Client Initialized.")
except:
    print("Error: Please set your GROQ_API_KEY in Colab Secrets.")

Groq Client Initialized.


In [57]:
# 1. Data Loading & Chunking
# Using the existing sample file if available, otherwise creating a mock
file_path = "faq.txt"
faq_content = """
Q: What is the company's remote work policy?
A: Employees can work remotely up to 3 days a week with manager approval.

Q: How do I request PTO?
A: PTO requests must be submitted through the HR portal at least 2 weeks in advance.

Q: What are the office hours?
A: Our core office hours are 9:00 AM to 5:00 PM local time.
"""
with open(file_path, "w") as f:
    f.write(faq_content)

loader = TextLoader(file_path)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

# 2. Embedding & Vector Store
# all-MiniLM-L6-v2 is small, fast, and runs locally in the notebook
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

print(f"Vectorstore created with {len(splits)} chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 1 chunks.


### Upload Your FAQ Document
Run the cell below to upload your own file. The system will process it and update the retrieval index.

In [58]:
from google.colab import files
import os

uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        file_path = filename
        print(f"Successfully uploaded: {filename}")

        # Process the new file
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        else:
            loader = TextLoader(file_path)

        new_docs = loader.load()
        new_splits = text_splitter.split_documents(new_docs)

        # Update the existing vectorstore with new documents
        vectorstore.add_documents(new_splits)
        print(f"Added {len(new_splits)} new chunks to the vector store.")
else:
    print("No file uploaded. Using the default mock FAQ.")

Saving Sample Company FAQ Document.pdf to Sample Company FAQ Document (1).pdf
Successfully uploaded: Sample Company FAQ Document (1).pdf
Added 9 new chunks to the vector store.


In [61]:
def rag_pipeline(query):
    # 1. Retrieval: Get top 2 relevant chunks
    docs = vectorstore.similarity_search_with_relevance_scores(query, k=2)

    context_text = "\n\n---\n\n".join([doc.page_content for doc, score in docs])

    # 2. Generation: Construct the prompt
    prompt = f"""You are a helpful HR Assistant. Use the provided context to answer the question.
    If the answer isn't in the context, say you don't know.

    Context:
    {context_text}

    Question: {query}

    Answer:"""

    # 3. Groq API Call - Updated to a supported model
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=500
    )

    return {
        "answer": completion.choices[0].message.content,
        "source_documents": docs
    }

print("RAG Pipeline Updated with llama-3.1-8b-instant.")

RAG Pipeline Updated with llama-3.1-8b-instant.


In [63]:
test_queries = [
    "What is the remote work policy?",
    "How do I request time off?",
    "Do we get free snacks?" # Out of scope test
]

for q in test_queries:
    print(f"\nQUERY: {q}")
    try:
        result = rag_pipeline(q)
        print(f"RESPONSE: {result['answer']}")
        # Display similarity score for debugging
        for doc, score in result['source_documents']:
            print(f"[Score: {score:.4f}] Source: {doc.metadata.get('source', 'Unknown')} | Content: {doc.page_content[:50]}...")
    except Exception as e:
        print(f"Error processing query: {e}")


QUERY: What is the remote work policy?
RESPONSE: Employees can work remotely up to 3 days a week with manager approval.
[Score: 0.4578] Source: faq.txt | Content: Q: What is the company's remote work policy?
A: Em...
[Score: -0.0622] Source: Sample Company FAQ Document (1).pdf | Content: generated
 
and
 
sent
 
to
 
your
 
registered
 
...

QUERY: How do I request time off?


/tmp/ipykernel_3826/2079529722.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'source': 'faq.txt'}, page_content="Q: What is the company's remote work policy?\nA: Employees can work remotely up to 3 days a week with manager approval.\n\nQ: How do I request PTO?\nA: PTO requests must be submitted through the HR portal at least 2 weeks in advance.\n\nQ: What are the office hours?\nA: Our core office hours are 9:00 AM to 5:00 PM local time."), 0.45779621965390327), (Document(metadata={'page_label': '1', 'source': 'Sample Company FAQ Document (1).pdf', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sample Company FAQ Document', 'producer': 'Skia/PDF m148 Google Docs Renderer', 'page': 0, 'total_pages': 3}, page_content='generated\n \nand\n \nsent\n \nto\n \nyour\n \nregistered\n \nemail.\n \n \nTechnical  Support'), -0.062244050346860647)]
  docs = vectorstore.similarity_search_with_relevance_scores(query, k=2)


RESPONSE: PTO requests must be submitted through the HR portal at least 2 weeks in advance.
[Score: 0.1503] Source: faq.txt | Content: Q: What is the company's remote work policy?
A: Em...
[Score: 0.0403] Source: Sample Company FAQ Document (1).pdf | Content: Sample  Company  FAQ  Document  
General  Informat...

QUERY: Do we get free snacks?
RESPONSE: I don't know. The provided context does not mention anything about free snacks. It appears to be related to billing, payments, refunds, invoices, security, and data protection.
[Score: -0.2041] Source: Sample Company FAQ Document (1).pdf | Content: under
 
the
 
“Request
 
Demo”
 
section.
 
 
Bill...
[Score: -0.3207] Source: Sample Company FAQ Document (1).pdf | Content: your
 
account.
 
 
Security  and  Privacy  
Q16: ...


/tmp/ipykernel_3826/2079529722.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'total_pages': 3, 'page_label': '1', 'creationdate': '', 'source': 'Sample Company FAQ Document (1).pdf', 'creator': 'PyPDF', 'producer': 'Skia/PDF m148 Google Docs Renderer', 'page': 0, 'title': 'Sample Company FAQ Document'}, page_content='under\n \nthe\n \n“Request\n \nDemo”\n \nsection.\n \n \nBilling  and  Payments  \nQ7:  What  payment  methods  do  you  accept?  \nA:\n \nWe\n \naccept\n \ncredit\n \ncards,\n \ndebit\n \ncards,\n \nnet\n \nbanking,\n \nand\n \nUPI\n \npayments.\n \nQ8:  Do  you  offer  refunds?  \nA:\n \nRefunds\n \nare\n \navailable\n \nwithin\n \n14\n \ndays\n \nof\n \npurchase\n \nif\n \nthe\n \nservice\n \nhas\n \nnot\n \nbeen\n \nused.\n \nQ9:  Can  I  get  an  invoice  for  my  purchase?  \nA:\n \nYes,\n \ninvoices\n \nare\n \nautomatically\n \ngenerated\n \nand\n \nsent\n \nto\n \nyour'), -0.2041063806307699), (Document(metadata={'creator': 

# Part 5: Evaluation Methodology

To ensure this system is production-ready, we measure four key metrics:

1.  **Retrieval Accuracy:** Use the `similarity_search_with_relevance_scores` to ensure the correct chunk is consistently in the top results (Score > 0.7).
2.  **Response Relevance:** Verify that the LLM uses the context and doesn't hallucinate outside the provided facts.
3.  **Latency:** Monitor the time for the Groq API call (Llama-3 on Groq typically takes < 500ms).
4.  **Faithfulness:** Check if the answer contradicts any retrieved context.